# Macro regimes and market relationships

Combine FRED series with equity prices, build a simple monthly regime table, and ask Tyche for a server-computed rolling correlation.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tyche_client import TycheClient

from dotenv import load_dotenv

load_dotenv(override=True)  # reads TYCHE_BASE_URL / TYCHE_API_KEY from a local .env file (see .env.example)
# override=True: without it, editing .env and re-running this cell in an already-running
# kernel silently keeps the OLD value, since load_dotenv() won't clobber an env var that
# a previous run of this same cell already set -- a real bug this caused once, not hypothetical.

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Retrieve macro and market data

FRED IDs: `CPIAUCSL` (CPI), `FEDFUNDS` (policy rate), `UNRATE` (unemployment), and `DGS10` (10-year Treasury yield).

In [ ]:
start = "2015-01-01"
macro = pd.concat({
    "cpi": client.series("fred", "CPIAUCSL", start=start),
    "fed_funds": client.series("fred", "FEDFUNDS", start=start),
    "unemployment": client.series("fred", "UNRATE", start=start),
    "ten_year": client.series("fred", "DGS10", start=start),
}, axis=1).sort_index()

spy = client.prices("SPY", start=start)["Close"]
macro.tail()

## Build a monthly descriptive regime table

A month is labelled by whether year-over-year inflation is accelerating and whether the policy rate rose. This is deliberately descriptive; it does not imply causality.

In [ ]:
month_end = pd.offsets.MonthEnd()
monthly = macro.resample(month_end).last()
monthly["inflation_yoy"] = monthly["cpi"].pct_change(12) * 100
monthly["spy_return"] = spy.resample(month_end).last().pct_change()
inflation_change = monthly["inflation_yoy"].diff()
policy_change = monthly["fed_funds"].diff()
monthly["inflation_direction"] = inflation_change.gt(0).map({True: "accelerating", False: "cooling"}).where(inflation_change.notna())
monthly["policy_direction"] = policy_change.gt(0).map({True: "tightening", False: "not tightening"}).where(policy_change.notna())

regime_stats = monthly.groupby(["inflation_direction", "policy_direction"])["spy_return"].agg(["mean", "median", "std", "count"])
regime_stats.style.format({"mean": "{:.2%}", "median": "{:.2%}", "std": "{:.2%}"})

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
monthly["inflation_yoy"].plot(ax=axes[0], color="firebrick", title="CPI inflation (year over year)")
monthly[["fed_funds", "ten_year"]].plot(ax=axes[1], title="Policy rate and 10-year yield")
monthly["unemployment"].plot(ax=axes[2], color="slateblue", title="Unemployment rate")
plt.tight_layout()

## Server-side rolling relationship

Tyche aligns observation dates, converts both series to weekly percentage changes, and computes a one-year rolling Pearson correlation.

In [ ]:
rolling = client.rolling_correlation(
    "ticker", "SPY", "fred", "DGS10",
    start_date="2018-01-01",
    frequency="1w",
)

rolling["correlation"].plot(figsize=(11, 4), title="SPY vs 10-year yield: rolling correlation of weekly changes")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("correlation")
plt.show()

Try changing either side to another supported source, for example `("vix", "VIX")`, `("statcan", "v2062811")`, or `("wbnk", "USA:NY.GDP.MKTP.CD")`.